# LeetCode #1284: Minimum Number of Flips to Convert Binary Matrix to Zero Matrix

https://leetcode.com/problems/minimum-number-of-flips-to-convert-binary-matrix-to-zero-matrix/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (DFS all flip orders)** | $O((n \cdot m)! )$ | $O(n \cdot m)$ |
| **Optimal: BFS on Bitmask State Space ★** | $O(2^{n \cdot m} \cdot n \cdot m)$ | $O(2^{n \cdot m})$ |

---

## Understanding the Methods

### Brute Force (DFS all flip orders)
Try every possible order of flipping cells and track the minimum flips needed to reach the all-zero state. With up to $3 \times 3 = 9$ cells, this is $(n \cdot m)!$ paths — completely infeasible beyond tiny inputs.

### Optimal: BFS on Bitmask State Space ★
Encode the entire matrix as a single integer bitmask (at most $2^9 = 512$ states for $3 \times 3$). BFS from the initial state, exploring all reachable states by applying each of the $n \cdot m$ possible flips. Each flip toggles the bit for the chosen cell and all its orthogonal neighbors. BFS guarantees the first time the all-zero state (bitmask = 0) is reached is the minimum number of flips.

**Constraints:**
* m == mat.length
* n == mat[i].length
* 1 <= m, n <= 3
* mat[i][j] is 0 or 1

## Solutions
### C#

In [ ]:
// BFS on bitmask state: encode matrix as integer, explore all flip transitions
public class Solution {
    public int MinFlips(int[][] mat) {
        int m = mat.Length, n = mat[0].Length;
        int total = m * n;

        // Encode the initial matrix into a bitmask
        int start = 0;
        for (int r = 0; r < m; r++)
            for (int c = 0; c < n; c++)
                if (mat[r][c] == 1)
                    start |= 1 << (r * n + c);

        if (start == 0) return 0; // Already all zeros

        // Pre-compute the flip mask for each cell (cell + orthogonal neighbors)
        int[] flipMask = new int[total];
        int[] dr = { 0, 0, 1, -1 };
        int[] dc = { 1, -1, 0, 0 };
        for (int i = 0; i < total; i++) {
            int r = i / n, c = i % n;
            flipMask[i] = 1 << i; // The cell itself
            for (int d = 0; d < 4; d++) {
                int nr = r + dr[d], nc = c + dc[d];
                if (nr >= 0 && nr < m && nc >= 0 && nc < n)
                    flipMask[i] |= 1 << (nr * n + nc);
            }
        }

        // BFS: state = bitmask of the current matrix
        var visited = new HashSet<int> { start };
        var queue = new Queue<int>();
        queue.Enqueue(start);
        int steps = 0;

        while (queue.Count > 0) {
            steps++;
            int size = queue.Count;
            while (size-- > 0) {
                int state = queue.Dequeue();
                // Try every possible single-cell flip
                for (int i = 0; i < total; i++) {
                    int next = state ^ flipMask[i];
                    if (next == 0) return steps; // Reached the all-zero target
                    if (visited.Add(next))
                        queue.Enqueue(next);
                }
            }
        }
        return -1; // Unreachable
    }
}

### Python

In [ ]:
# BFS on bitmask state: encode matrix as integer, explore all flip transitions
from typing import List
from collections import deque

class Solution:
    def minFlips(self, mat: List[List[int]]) -> int:
        m, n = len(mat), len(mat[0])
        total = m * n

        # Encode the matrix into a starting bitmask
        start = 0
        for r in range(m):
            for c in range(n):
                if mat[r][c]:
                    start |= 1 << (r * n + c)

        if start == 0:
            return 0  # Already the zero matrix

        # Pre-compute the XOR mask for each flip (cell + orthogonal neighbors)
        flip_masks = []
        for i in range(total):
            r, c = divmod(i, n)
            mask = 1 << i
            for dr, dc in ((0,1),(0,-1),(1,0),(-1,0)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    mask |= 1 << (nr * n + nc)
            flip_masks.append(mask)

        # BFS over bitmask states
        visited = {start}
        queue = deque([start])
        steps = 0

        while queue:
            steps += 1
            for _ in range(len(queue)):
                state = queue.popleft()
                for mask in flip_masks:
                    nxt = state ^ mask
                    if nxt == 0:
                        return steps
                    if nxt not in visited:
                        visited.add(nxt)
                        queue.append(nxt)

        return -1

### Go

In [ ]:
// BFS on bitmask state: encode matrix as integer, explore all flip transitions
package main

func minFlips(mat [][]int) int {
    m, n := len(mat), len(mat[0])
    total := m * n

    // Encode initial state into a bitmask
    start := 0
    for r := 0; r < m; r++ {
        for c := 0; c < n; c++ {
            if mat[r][c] == 1 {
                start |= 1 << (r*n + c)
            }
        }
    }
    if start == 0 {
        return 0
    }

    // Compute flip masks: each flip toggles a cell and its orthogonal neighbors
    dirs := [][2]int{{0,1},{0,-1},{1,0},{-1,0}}
    flipMasks := make([]int, total)
    for i := 0; i < total; i++ {
        r, c := i/n, i%n
        flipMasks[i] = 1 << i
        for _, d := range dirs {
            nr, nc := r+d[0], c+d[1]
            if nr >= 0 && nr < m && nc >= 0 && nc < n {
                flipMasks[i] |= 1 << (nr*n + nc)
            }
        }
    }

    // BFS: shortest path to the all-zero state
    visited := map[int]bool{start: true}
    queue := []int{start}
    steps := 0

    for len(queue) > 0 {
        steps++
        size := len(queue)
        for i := 0; i < size; i++ {
            state := queue[i]
            for _, mask := range flipMasks {
                next := state ^ mask
                if next == 0 {
                    return steps
                }
                if !visited[next] {
                    visited[next] = true
                    queue = append(queue, next)
                }
            }
        }
        queue = queue[size:]
    }
    return -1
}

### Rust

In [ ]:
// BFS on bitmask state: encode matrix as integer, explore all flip transitions
use std::collections::{HashSet, VecDeque};

impl Solution {
    pub fn min_flips(mat: Vec<Vec<i32>>) -> i32 {
        let (m, n) = (mat.len(), mat[0].len());
        let total = m * n;

        // Encode the initial matrix as a bitmask
        let mut start = 0usize;
        for r in 0..m {
            for c in 0..n {
                if mat[r][c] == 1 {
                    start |= 1 << (r * n + c);
                }
            }
        }
        if start == 0 { return 0; }

        // Pre-compute flip masks: cell + orthogonal neighbors
        let dirs: [(isize, isize); 4] = [(0,1),(0,-1),(1,0),(-1,0)];
        let flip_masks: Vec<usize> = (0..total).map(|i| {
            let (r, c) = (i / n, i % n);
            let mut mask = 1 << i;
            for &(dr, dc) in &dirs {
                let (nr, nc) = (r as isize + dr, c as isize + dc);
                if nr >= 0 && nr < m as isize && nc >= 0 && nc < n as isize {
                    mask |= 1 << (nr as usize * n + nc as usize);
                }
            }
            mask
        }).collect();

        // BFS over reachable bitmask states
        let mut visited: HashSet<usize> = HashSet::new();
        visited.insert(start);
        let mut queue: VecDeque<usize> = VecDeque::new();
        queue.push_back(start);
        let mut steps = 0i32;

        while !queue.is_empty() {
            steps += 1;
            let size = queue.len();
            for _ in 0..size {
                let state = queue.pop_front().unwrap();
                for &mask in &flip_masks {
                    let next = state ^ mask;
                    if next == 0 { return steps; }
                    if visited.insert(next) {
                        queue.push_back(next);
                    }
                }
            }
        }
        -1
    }
}

## Example Scenarios

**1. Common Case** — Single 1 in the center

**Input:** `mat = [[0,0],[0,1]]`
Start bitmask = 0b1000 (bit 3 set). One flip of cell (1,1) toggles bits for (1,0), (0,1), and (1,1). After that flip the state becomes 0b0110. Two more flips of (1,0) and (0,1) clear those bits. BFS finds the minimum at 3 flips. Returns `3`.

**2. Slightly Complex** — All zeros already

**Input:** `mat = [[0,0],[0,0]]`
Start bitmask = 0. Immediately returns `0` before even entering BFS.

**3. Edge Case: Time Factor** — 3×3 all-ones matrix

**Input:** `mat = [[1,1,1],[1,1,1],[1,1,1]]`
Start bitmask = 0b111111111 = 511. BFS explores up to $2^9 = 512$ distinct states. Each state tries 9 flips. Total work is bounded by $512 \times 9 = 4608$ operations — trivially fast.

**4. Edge Case: Space Factor** — 3×3 with an unreachable zero state

**Input:** `mat = [[1,0,0],[0,0,0],[0,0,0]]`
BFS exhausts all $2^9 = 512$ reachable states from the start. If none equals 0, returns -1. The visited set holds at most 512 entries — $O(2^{nm})$ space at most.

**5. Almost-Impossible but Plausible** — Checkerboard pattern

**Input:** `mat = [[1,0,1],[0,1,0],[1,0,1]]`
Bitmask = 0b101010101 = 341. Each flip of a corner cell toggles 3 bits (itself + 2 neighbors); center toggles 5. BFS must explore many intermediate states before reaching 0. Even so, all $\leq 512$ states are processed within microseconds.